<a href="https://colab.research.google.com/github/Atovenish/THO.github.io/blob/main/HandsOn-2-2-LikelihoodFitting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Only if you are using colab, you need to mount your drive and go to the directory where the necessary files are located : StatsDataAna

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
cd /content/drive/My Drive/StatsDataAna

[Errno 2] No such file or directory: '/content/drive/My Drive/StatsDataAna'
/content


Additionally, you need to install ROOT software in Colab

In [7]:
!pip install root

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.9/247.9 MB 3.3 MB/s eta 0:00:00


Binned maximum likelihood fitting is much faster than unbinned maximum likelihood fitting.

We will use histograms as PDF yielding n = (n1,n2,n3,...,nN) in N bins.

In [11]:
#we will rely on ROOT data analysis framework for fitting
import ROOT

/usr/local/lib/python3.12/dist-packages/ROOT/__init__.py:222: UserWarning: 
This distribution of ROOT is in alpha stage. Feedback is welcome and appreciated. Feel free to reach out to the user forum for questions and general feedback at https://root-forum.cern.ch or to submit an issue at https://github.com/root-project/root/issues. Do not rely on this distribution for production purposes.

  warnings.warn(


In [12]:
from ROOT import TChain, TFile, TH1F, TCanvas
from math import *

names = ["ttbar","wjets","dy","data"]

for name in names:
    # open the file
    chain = TChain("events");
    chain.Add('files/'+name+'.root')

    f = TFile('hist_'+name+'.root', 'RECREATE')

    h_njets = TH1F('h_njets','number of jets',10,0.0,10.0)


    entries = chain.GetEntries()
    for i in range(entries):
        chain.GetEntry(i)
        #met = math.sqrt( chain.MET_px**2 + chain.MET_py**2 ) # we can use missing energy to remove Z boson events
        if chain.NMuon > 1:
            h_njets.Fill(chain.NJet, chain.EventWeight)

    ntotal = h_njets.Integral()
    print ("total number of events from ", name , "= ", "%.1f" % ntotal, "(raw events = ", entries , ")")

    f.Write()
    f.Close()

Error in <TFile::TFile>: file /content/files/ttbar.root does not exist
Error in <TFile::TFile>: file /content/files/wjets.root does not exist
Error in <TFile::TFile>: file /content/files/dy.root does not exist
Error in <TFile::TFile>: file /content/files/data.root does not exist


total number of events from  ttbar =  0.0 (raw events =  0 )
total number of events from  wjets =  0.0 (raw events =  0 )
total number of events from  dy =  0.0 (raw events =  0 )
total number of events from  data =  0.0 (raw events =  0 )


In [13]:
ls *.root

hist_data.root  hist_dy.root  hist_ttbar.root  hist_wjets.root


##### answer :

```
hist_data.root  hist_dy.root  hist_ttbar.root  hist_wjets.root
```

Now we will read the histograms.

In [ ]:
from ROOT import TChain, TFile, TH1F, TCanvas

f_data = TFile("hist_data.root")
f_dy = TFile("hist_dy.root")
f_wjets = TFile("hist_wjets.root")
f_ttbar = TFile("hist_ttbar.root")

h_data = f_data.Get("h_njets")
h_dy = f_dy.Get("h_njets")
h_wjets = f_wjets.Get("h_njets")
h_ttbar = f_ttbar.Get("h_njets")

And check how many bins we have in the histogram.

In [ ]:
nbins = h_data.GetXaxis().GetNbins()
print (nbins)

##### answer:

```
10
```

We will use RooStats for the binned likelihood fitting. We don't need to worry about this part.
We will just import necessary library and set the variable and histograms in this RooStats format.  

In [ ]:
from ROOT import RooRealVar, RooDataHist, RooArgList
x = RooRealVar("x","x", 0, 10)
data = RooDataHist("data","data", ROOT.RooArgList(x), h_data)
ttbar = RooDataHist("ttbar","ttbar", ROOT.RooArgList(x), h_ttbar)
dy = RooDataHist("zjets","zjets", ROOT.RooArgList(x), h_dy)
wjets = RooDataHist("wjets","wjets", ROOT.RooArgList(x), h_wjets)

We will need to make templates for each process.

In [ ]:
from ROOT import RooHistPdf
pdf_ttbar = RooHistPdf("pdf_ttbar","pdf_ttbar", ROOT.RooArgSet(x), ttbar)
pdf_dy = RooHistPdf("pdf_dy","pdf_dy", ROOT.RooArgSet(x), dy)
pdf_wjets = RooHistPdf("pdf_wjets","pdf_wjets", ROOT.RooArgSet(x), wjets)

create parameter and initialize it

In [ ]:
nttbar = RooRealVar("nttbar","nttbar",200, 0, 10000)
ndy = RooRealVar("ndy","ndy", 20000 , 0, 40000)
nwjets = RooRealVar("nwjets","nwjets", 300 , 0, 10000)

create a model

extended likelihood model below

$$ M(x) = N_{ttbar} \cdot S_{ttbar} (x) + N_{dy} \cdot B_{dy} (x) + N_{wjets} \cdot B_{wjets} (x)  $$

In [ ]:
from ROOT import RooAddPdf
model = RooAddPdf("model","model",RooArgList(pdf_ttbar, pdf_dy, pdf_wjets), RooArgList(nttbar, ndy, nwjets))

perform the fit

In [ ]:
fitResult = model.fitTo( data );

We will visualize the fit result. Each histogram will have the post fit number for normalization.

In [ ]:
c = TCanvas("c","c",1)
xframe = x.frame()
data.plotOn(xframe)
model.paramOn(xframe, ROOT.RooFit.Layout(0.65,0.9,0.9) )
model.plotOn(xframe,ROOT.RooFit.Components("pdf_wjets,pdf_dy,pdf_ttbar"),ROOT.RooFit.LineColor(2),ROOT.RooFit.FillColor(2),ROOT.RooFit.DrawOption("F"))
model.plotOn(xframe,ROOT.RooFit.Components("pdf_wjets,pdf_dy"),ROOT.RooFit.LineColor(5),ROOT.RooFit.FillColor(5),ROOT.RooFit.DrawOption("F"))
model.plotOn(xframe,ROOT.RooFit.Components("pdf_wjets"),ROOT.RooFit.LineColor(3),ROOT.RooFit.FillColor(3),ROOT.RooFit.DrawOption("F"))
model.plotOn(xframe)
data.plotOn(xframe)
xframe.Draw()
c.SetLogy()
c.Draw()

##### task

Can you wriate a macro to fit again including single_top process this time?

### Re-fitting with 'single_top' process

We will now extend the previous fitting procedure to include the 'single_top' process. This involves generating the histogram for 'single_top', updating the model, and performing a new fit.

In [ ]:
from ROOT import TChain, TFile, TH1F
from math import *

# Include 'single_top' in the list of processes
names_extended = ["ttbar","wjets","dy","data", "single_top"]

print("Re-generating histograms for all processes, including single_top...")
for name in names_extended:
    # open the file
    chain = TChain("events");
    # Assuming 'files/single_top.root' exists in the same location as others
    chain.Add('files/'+name+'.root')

    f = TFile('hist_'+name+'.root', 'RECREATE')

    h_njets = TH1F('h_njets','number of jets',10,0.0,10.0)

    entries = chain.GetEntries()
    for i in range(entries):
        chain.GetEntry(i)
        if chain.NMuon > 1:
            h_njets.Fill(chain.NJet, chain.EventWeight)

    ntotal = h_njets.Integral()
    print (f"Total number of events from {name} = {ntotal:.1f} (raw events = {entries})")

    f.Write()
    f.Close()

print("Histograms re-generated successfully.")

In [ ]:
from ROOT import TFile

# Re-read all histograms, including single_top
f_data = TFile("hist_data.root")
f_dy = TFile("hist_dy.root")
f_wjets = TFile("hist_wjets.root")
f_ttbar = TFile("hist_ttbar.root")
f_singletop = TFile("hist_single_top.root")

h_data = f_data.Get("h_njets")
h_dy = f_dy.Get("h_njets")
h_wjets = f_wjets.Get("h_njets")
h_ttbar = f_ttbar.Get("h_njets")
h_singletop = f_singletop.Get("h_njets")

print("Histograms re-loaded.")

In [ ]:
from ROOT import RooRealVar, RooDataHist, RooArgList

# Assuming 'x' RooRealVar is already defined from previous cells
# x = RooRealVar("x","x", 0, 10)

data = RooDataHist("data","data", ROOT.RooArgList(x), h_data)
ttbar = RooDataHist("ttbar","ttbar", ROOT.RooArgList(x), h_ttbar)
dy = RooDataHist("zjets","zjets", ROOT.RooArgList(x), h_dy)
wjets = RooDataHist("wjets","wjets", ROOT.RooArgList(x), h_wjets)
singletop = RooDataHist("singletop","singletop", ROOT.RooArgList(x), h_singletop)

print("RooDataHist objects created for all processes.")

In [ ]:
from ROOT import RooHistPdf, RooArgSet

pdf_ttbar = RooHistPdf("pdf_ttbar","pdf_ttbar", ROOT.RooArgSet(x), ttbar)
pdf_dy = RooHistPdf("pdf_dy","pdf_dy", ROOT.RooArgSet(x), dy)
pdf_wjets = RooHistPdf("pdf_wjets","pdf_wjets", ROOT.RooArgSet(x), wjets)
pdf_singletop = RooHistPdf("pdf_singletop","pdf_singletop", ROOT.RooArgSet(x), singletop)

print("RooHistPdf objects created for all processes.")

In [ ]:
from ROOT import RooRealVar

nttbar = RooRealVar("nttbar","nttbar",200, 0, 10000)
ndy = RooRealVar("ndy","ndy", 20000 , 0, 40000)
nwjets = RooRealVar("nwjets","nwjets", 300 , 0, 10000)
nsingletop = RooRealVar("nsingletop","nsingletop", 50, 0, 5000) # Initial guess for single_top events

print("Parameters (RooRealVar) initialized for all processes.")

### Updated Model with 'single_top'

$$ M(x) = N_{ttbar} \cdot S_{ttbar} (x) + N_{dy} \cdot B_{dy} (x) + N_{wjets} \cdot B_{wjets} (x) + N_{singletop} \cdot S_{singletop} (x)  $$

In [ ]:
from ROOT import RooAddPdf, RooArgList

# Create the extended model including single_top
model_extended = RooAddPdf("model_extended","model_extended",
                           RooArgList(pdf_ttbar, pdf_dy, pdf_wjets, pdf_singletop),
                           RooArgList(nttbar, ndy, nwjets, nsingletop))

print("Extended model created successfully.")

In [ ]:
# Perform the fit with the extended model
print("Performing fit with extended model...")
fitResult_extended = model_extended.fitTo( data );
print("Fit complete.")

In [ ]:
from ROOT import TCanvas

# Visualize the fit result for the extended model
c_extended = TCanvas("c_extended","Extended Fit Result",1)
xframe_extended = x.frame()
data.plotOn(xframe_extended)
model_extended.paramOn(xframe_extended, ROOT.RooFit.Layout(0.65,0.9,0.9) )

# Plot components in layers, ensuring proper stacking order
model_extended.plotOn(xframe_extended,ROOT.RooFit.Components("pdf_singletop,pdf_wjets,pdf_dy,pdf_ttbar"),ROOT.RooFit.LineColor(2),ROOT.RooFit.FillColor(2),ROOT.RooFit.DrawOption("F")) # ttbar
model_extended.plotOn(xframe_extended,ROOT.RooFit.Components("pdf_singletop,pdf_wjets,pdf_dy"),ROOT.RooFit.LineColor(5),ROOT.RooFit.FillColor(5),ROOT.RooFit.DrawOption("F")) # dy
model_extended.plotOn(xframe_extended,ROOT.RooFit.Components("pdf_singletop,pdf_wjets"),ROOT.RooFit.LineColor(3),ROOT.RooFit.FillColor(3),ROOT.RooFit.DrawOption("F")) # wjets
model_extended.plotOn(xframe_extended,ROOT.RooFit.Components("pdf_singletop"),ROOT.RooFit.LineColor(4),ROOT.RooFit.FillColor(4),ROOT.RooFit.DrawOption("F")) # single_top (new color)

# Plot the full model and data again on top
model_extended.plotOn(xframe_extended)
data.plotOn(xframe_extended)

xframe_extended.Draw()
c_extended.SetLogy()
c_extended.Draw()

print("Extended fit visualization complete.")